In [1]:
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')

import numpy as np
import pandas as pd
from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client

db = DuckDBClient()
s  = S3Client()

# Load HDFCBANK + BANKNIFTY for all available dates
symbols = {
    'HDFCBANK':  ['HDFCBANK26SEPFUT', 'HDFCBANK26OCTFUT'],
    'BANKNIFTY': ['BANKNIFTY26SEPFUT', 'BANKNIFTY26OCTFUT'],
    'ICICIBANK': ['ICICIBANK26SEPFUT', 'ICICIBANK26OCTFUT'],
}

data = {}
for name, syms in symbols.items():
    all_df = []
    for sym in syms:
        files = s.list_files(f'processed/1s/{sym}/')
        for f in files:
            date = f.split('/')[-1].replace('.parquet','')
            try:
                df = db.read_parquet(f'processed/1s/{sym}/{date}.parquet')
                df['ist_sec'] = (df['ts_sec'] + 19800) % 86400
                df = df[(df['ist_sec'] >= 9*3600+15*60) &
                        (df['ist_sec'] <= 15*3600+30*60)].reset_index(drop=True)
                df['date']   = date
                df['symbol'] = name
                all_df.append(df)
            except:
                continue
    if all_df:
        combined = pd.concat(all_df, ignore_index=True)
        data[name] = combined
        print(f'{name}: {len(combined):,} bars across {combined["date"].nunique()} days')

db.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BANKNIFTY: 111,695 bars across 7 days


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ICICIBANK: 111,376 bars across 7 days


In [2]:
# ── Option B: Momentum signal for HDFCBANK ────────────────────────
TICK = 0.05
W_sig = 30
NW    = 300

def build_signals(df, tick=0.05, mean_reversion=True):
    sig_dfs = []
    for d in sorted(df['date'].unique()):
        sub = df[df['date']==d].copy().reset_index(drop=True)
        
        sub['bid_chg']    = sub['total_bid_qty'].diff()
        sub['ask_chg']    = sub['total_ask_qty'].diff()
        pu                = sub['close'].diff().abs() < tick*0.5
        sub['cancel_imb'] = 0.0
        sub.loc[pu&(sub['bid_chg']<0),'cancel_imb'] -= sub['bid_chg'].abs()
        sub.loc[pu&(sub['ask_chg']<0),'cancel_imb'] += sub['ask_chg'].abs()

        sub['s_cancel']   = -sub['cancel_imb'].rolling(W_sig).mean()
        sub['s_imb']      =  sub['imbalance_last'].rolling(W_sig).mean()
        sub['mid']        = (sub['bid_p1']+sub['ask_p1'])/2
        sub['s_wmid']     = (sub['weighted_mid']-sub['mid']).rolling(10).mean()
        sub['s_momentum'] =  sub['close'].pct_change(30)

        for col in ['s_cancel','s_imb','s_wmid','s_momentum']:
            rs       = sub[col].rolling(NW).std()+1e-9
            sub[col] = (sub[col]/rs).clip(-1,1)

        if mean_reversion:
            # USDINR style — fade the move
            sub['score'] = (0.393*sub['s_cancel'] - 0.218*sub['s_imb']
                          - 0.320*sub['s_wmid']   - 0.069*sub['s_momentum'])
        else:
            # Momentum style — follow the move
            sub['score'] = (-0.393*sub['s_cancel'] + 0.218*sub['s_imb']
                           + 0.320*sub['s_wmid']   + 0.400*sub['s_momentum'])

        for h in [60, 300, 1800]:
            sub[f'fwd_{h}s'] = sub['close'].shift(-h) - sub['close']

        sub['oi_chg']    = sub['oi'].diff()
        sub['oi_chg_pct']= sub['oi'].pct_change(60)
        sub['hour']      = sub['ist_sec'] // 3600
        sig_dfs.append(sub)

    return pd.concat(sig_dfs, ignore_index=True).reset_index(drop=True)

# Build both signal types for HDFCBANK
df_hdfc_mr  = build_signals(data['HDFCBANK'], tick=0.05, mean_reversion=True)
df_hdfc_mom = build_signals(data['HDFCBANK'], tick=0.05, mean_reversion=False)

print('=== Option B: Mean Reversion vs Momentum Signal ===')
print(f'{"Signal":<20} {"IC_60s":>8} {"IC_300s":>8} {"IC_1800s":>8}')
print('-'*48)
for name, df_s in [('MeanReversion', df_hdfc_mr), ('Momentum', df_hdfc_mom)]:
    val = df_s.dropna(subset=['score','fwd_1800s'])
    ic60   = val['score'].corr(val['fwd_60s'])
    ic300  = val['score'].corr(val['fwd_300s'])
    ic1800 = val['score'].corr(val['fwd_1800s'])
    print(f'{name:<20} {ic60:>8.4f} {ic300:>8.4f} {ic1800:>8.4f}')

# Check by hour
print()
print('=== Momentum Signal IC by Hour ===')
print(f'{"Hour":<6} {"IC_300s":>8} {"IC_1800s":>8} {"n":>8}')
print('-'*32)
val = df_hdfc_mom.dropna(subset=['score','fwd_1800s'])
for h in range(9, 16):
    mask  = val['hour'] == h
    ic300  = val[mask]['score'].corr(val[mask]['fwd_300s'])
    ic1800 = val[mask]['score'].corr(val[mask]['fwd_1800s'])
    print(f'{h}:00   {ic300:>8.4f} {ic1800:>8.4f} {mask.sum():>8}')

=== Option B: Mean Reversion vs Momentum Signal ===
Signal                 IC_60s  IC_300s IC_1800s
------------------------------------------------
MeanReversion         -0.0340  -0.0366  -0.0833
Momentum              -0.0088   0.0085   0.0511

=== Momentum Signal IC by Hour ===
Hour    IC_300s IC_1800s        n
--------------------------------
9:00    -0.0571  -0.0437    10146
10:00     0.0261   0.1026    20152
11:00    -0.0080   0.0356    20103
12:00    -0.0448  -0.1486    19946
13:00    -0.0653  -0.0339    20202
14:00     0.1248   0.1129    17774
15:00        nan      nan        0


In [4]:
# ── Option C: OI Analysis ─────────────────────────────────────────
print('=== Option C: OI Change Analysis ===')
print()

# OI change combinations with price direction
# Classic OI interpretation:
#   Price UP + OI UP   = new longs (bullish)
#   Price UP + OI DOWN = short covering (weakening bullish)
#   Price DOWN + OI UP = new shorts (bearish)
#   Price DOWN + OI DOWN = long unwinding (weakening bearish)

df_oi = df_hdfc_mr.copy()
df_oi['price_chg_60']  = df_oi['close'].diff(60)
df_oi['oi_chg_60']     = df_oi['oi'].diff(60)
df_oi['price_chg_300'] = df_oi['close'].diff(300)
df_oi['oi_chg_300']    = df_oi['oi'].diff(300)

# Normalize
df_oi['price_chg_z'] = (df_oi['price_chg_60'] /
                         (df_oi['price_chg_60'].rolling(300).std() + 1e-9))
df_oi['oi_chg_z']    = (df_oi['oi_chg_60'] /
                         (df_oi['oi_chg_60'].rolling(300).std() + 1e-9))

# OI signal
df_oi['oi_signal'] = df_oi['price_chg_z'] * df_oi['oi_chg_z']

val = df_oi.dropna(subset=['oi_signal','fwd_1800s','fwd_300s'])

print(f'OI signal IC at 300s:  {val["oi_signal"].corr(val["fwd_300s"]):.4f}')
print(f'OI signal IC at 1800s: {val["oi_signal"].corr(val["fwd_1800s"]):.4f}')

# Classify OI regime
df_oi['oi_regime'] = np.where(
    (df_oi['price_chg_60'] > 0) & (df_oi['oi_chg_60'] > 0), 'LONG_BUILD',
    np.where(
        (df_oi['price_chg_60'] < 0) & (df_oi['oi_chg_60'] > 0), 'SHORT_BUILD',
        np.where(
            (df_oi['price_chg_60'] > 0) & (df_oi['oi_chg_60'] < 0), 'SHORT_COVER',
            'LONG_UNWIND'
        )
    )
)

print()
print(f'{"OI Regime":<15} {"Count":>8} {"Pct":>6} {"Fwd_1800s_mean":>16} {"IC_1800s":>10}')
print('-'*58)
for regime in ['LONG_BUILD','SHORT_BUILD','SHORT_COVER','LONG_UNWIND']:
    sub  = df_oi[df_oi['oi_regime']==regime].dropna(subset=['fwd_1800s'])
    ic   = sub['oi_signal'].corr(sub['fwd_1800s']) if len(sub) > 100 else np.nan
    print(f'{regime:<15} {len(sub):>8} {len(sub)/len(df_oi)*100:>5.1f}% '
          f'{sub["fwd_1800s"].mean():>16.4f} {ic:>10.4f}')

# OI + Momentum combined
print()
print('=== OI + Momentum Combined Signal ===')
df_oi['combined'] = (0.6*df_oi['s_momentum'] + 0.4*df_oi['oi_signal'])
df_oi['combined'] = (df_oi['combined'] /
                     (df_oi['combined'].rolling(300).std()+1e-9)).clip(-1,1)
val2 = df_oi.dropna(subset=['combined','fwd_1800s'])
print(f'Combined IC at 300s:  {val2["combined"].corr(val2["fwd_300s"]):.4f}')
print(f'Combined IC at 1800s: {val2["combined"].corr(val2["fwd_1800s"]):.4f}')

print()
print(f'{"Hour":<6} {"OI_IC":>8} {"Mom_IC":>8} {"Combined_IC":>12}')
print('-'*36)
for h in range(9, 15):
    mask    = val2['hour'] == h
    oi_ic   = val2[mask]['oi_signal'].corr(val2[mask]['fwd_1800s'])
    mom_ic  = val2[mask]['s_momentum'].corr(val2[mask]['fwd_1800s'])
    comb_ic = val2[mask]['combined'].corr(val2[mask]['fwd_1800s'])
    print(f'{h}:00   {oi_ic:>8.4f} {mom_ic:>8.4f} {comb_ic:>12.4f}')

=== Option C: OI Change Analysis ===

OI signal IC at 300s:  -0.0082
OI signal IC at 1800s: -0.0203

OI Regime          Count    Pct   Fwd_1800s_mean   IC_1800s
----------------------------------------------------------
LONG_BUILD         29966  24.3%          -0.0907    -0.0237
SHORT_BUILD        33136  26.9%           0.0196    -0.0553
SHORT_COVER        20002  16.2%           0.1586     0.0056
LONG_UNWIND        27522  22.3%           0.1773    -0.0020

=== OI + Momentum Combined Signal ===
Combined IC at 300s:  -0.0430
Combined IC at 1800s: -0.0153

Hour      OI_IC   Mom_IC  Combined_IC
------------------------------------
9:00     0.0087  -0.0804      -0.0451
10:00    -0.0140  -0.0306      -0.0063
11:00    -0.0464  -0.0451      -0.0745
12:00     0.0383  -0.0634      -0.0143
13:00    -0.0440  -0.0781      -0.0518
14:00     0.0529   0.0345       0.0593


In [5]:
# ── Option D: Basis Analysis ──────────────────────────────────────
# Futures basis = futures price - theoretical fair value
# For equity: Fair value = Spot × (1 + r×T) - dividends
# We don't have spot directly but can approximate:
#   Use VWAP as proxy for fair value
#   Or use cash market data

print('=== Option D: Basis Analysis ===')
print()

# Proxy basis: futures close vs VWAP
# When close > vwap: futures trading at premium to avg price
# When close < vwap: futures trading at discount
df_basis = df_hdfc_mr.copy()

# Basis 1: price vs vwap
df_basis['basis_vwap']   = df_basis['close'] - df_basis['vwap']
df_basis['basis_vwap_z'] = (df_basis['basis_vwap'] /
                             (df_basis['basis_vwap'].rolling(300).std()+1e-9))

# Basis 2: weighted mid vs mid
df_basis['mid']          = (df_basis['bid_p1'] + df_basis['ask_p1']) / 2
df_basis['basis_wmid']   = df_basis['weighted_mid'] - df_basis['mid']
df_basis['basis_wmid_z'] = (df_basis['basis_wmid'] /
                             (df_basis['basis_wmid'].rolling(300).std()+1e-9))

# Basis 3: price momentum vs OI momentum (divergence)
df_basis['price_mom_z']  = (df_basis['close'].pct_change(300) /
                             (df_basis['close'].pct_change(300).rolling(300).std()+1e-9))
df_basis['oi_mom_z']     = (df_basis['oi'].pct_change(300) /
                             (df_basis['oi'].pct_change(300).rolling(300).std()+1e-9))
df_basis['divergence']   = df_basis['price_mom_z'] - df_basis['oi_mom_z']

val = df_basis.dropna(subset=['basis_vwap_z','fwd_1800s'])

print(f'{"Signal":<20} {"IC_300s":>8} {"IC_1800s":>8}')
print('-'*38)
for sig in ['basis_vwap_z','basis_wmid_z','divergence','price_mom_z','oi_mom_z']:
    v    = df_basis.dropna(subset=[sig,'fwd_1800s'])
    ic300  = v[sig].corr(v['fwd_300s'])
    ic1800 = v[sig].corr(v['fwd_1800s'])
    print(f'{sig:<20} {ic300:>8.4f} {ic1800:>8.4f}')

# Check basis_wmid by hour (this is our best signal candidate)
print()
print('=== Basis WMID by Hour ===')
print(f'{"Hour":<6} {"IC_300s":>8} {"IC_1800s":>8} {"n":>8}')
print('-'*32)
for h in range(9, 15):
    mask   = val['hour'] == h
    ic300  = val[mask]['basis_wmid_z'].corr(val[mask]['fwd_300s'])
    ic1800 = val[mask]['basis_wmid_z'].corr(val[mask]['fwd_1800s'])
    print(f'{h}:00   {ic300:>8.4f} {ic1800:>8.4f} {mask.sum():>8}')

# Best combined signal for HDFCBANK
print()
print('=== Best Combined Signal for HDFCBANK ===')
df_basis['best_signal'] = (0.5*df_basis['basis_wmid_z'] +
                            0.3*df_basis['price_mom_z'] +
                            0.2*df_basis['oi_mom_z'])
df_basis['best_signal'] = (df_basis['best_signal'] /
                            (df_basis['best_signal'].rolling(300).std()+1e-9)).clip(-1,1)

v2 = df_basis.dropna(subset=['best_signal','fwd_1800s'])
print(f'Best combined IC at 300s:  {v2["best_signal"].corr(v2["fwd_300s"]):.4f}')
print(f'Best combined IC at 1800s: {v2["best_signal"].corr(v2["fwd_1800s"]):.4f}')

print()
print(f'{"Hour":<6} {"IC_1800s":>8} {"n":>8}')
print('-'*24)
for h in range(9, 15):
    mask   = v2['hour'] == h
    ic1800 = v2[mask]['best_signal'].corr(v2[mask]['fwd_1800s'])
    print(f'{h}:00   {ic1800:>8.4f} {mask.sum():>8}')

=== Option D: Basis Analysis ===

Signal                IC_300s IC_1800s
--------------------------------------
basis_vwap_z          -0.0678  -0.1406
basis_wmid_z           0.0160   0.0277
divergence             0.0128  -0.0052
price_mom_z           -0.0387  -0.0751
oi_mom_z              -0.0248  -0.0169

=== Basis WMID by Hour ===
Hour    IC_300s IC_1800s        n
--------------------------------
9:00     0.0013  -0.0659    12103
10:00     0.0195   0.0416    20199
11:00     0.0189   0.0583    20103
12:00     0.0048  -0.0865    19946
13:00    -0.0553  -0.0623    20202
14:00     0.0898   0.1209    17774

=== Best Combined Signal for HDFCBANK ===
Best combined IC at 300s:  -0.0499
Best combined IC at 1800s: -0.0533

Hour   IC_1800s        n
------------------------
9:00    -0.1239    11504
10:00    -0.2182    20199
11:00     0.1238    20103
12:00    -0.1415    19946
13:00    -0.2336    20202
14:00     0.1543    17774


In [6]:
# ── Full horizon analysis for HDFCBANK ────────────────────────────
horizons = [60, 120, 300, 600, 900, 1800, 3600]

# Build forward returns for all horizons
for h in horizons:
    df_basis[f'fwd_{h}s'] = df_basis.groupby('date')['close'].transform(
        lambda x: x.shift(-h) - x
    )

print('=== Full Horizon IC Analysis — HDFCBANK ===')
print()

signals = {
    'basis_wmid_z':  'Weighted Mid Basis',
    'price_mom_z':   'Price Momentum',
    'oi_mom_z':      'OI Momentum',
    'divergence':    'Price-OI Divergence',
}

print(f'{"Signal":<22}', end='')
for h in horizons:
    print(f'{h}s'.rjust(7), end='')
print()
print('-'*70)

for sig, name in signals.items():
    print(f'{name:<22}', end='')
    for h in horizons:
        v   = df_basis.dropna(subset=[sig, f'fwd_{h}s'])
        ic  = v[sig].corr(v[f'fwd_{h}s'])
        print(f'{ic:>7.3f}', end='')
    print()

# By hour for best signal (basis_wmid_z)
print()
print('=== basis_wmid_z IC by Hour and Horizon ===')
print(f'{"Hour":<6}', end='')
for h in horizons:
    print(f'{h}s'.rjust(8), end='')
print()
print('-'*62)

v = df_basis.dropna(subset=['basis_wmid_z'])
for hour in range(9, 15):
    print(f'{hour}:00  ', end='')
    for h in horizons:
        sub = v[v['hour']==hour].dropna(subset=[f'fwd_{h}s'])
        ic  = sub['basis_wmid_z'].corr(sub[f'fwd_{h}s'])
        print(f'{ic:>8.3f}', end='')
    print()

# Also check momentum signal
print()
print('=== s_momentum IC by Hour and Horizon ===')
print(f'{"Hour":<6}', end='')
for h in horizons:
    print(f'{h}s'.rjust(8), end='')
print()
print('-'*62)

for hour in range(9, 15):
    print(f'{hour}:00  ', end='')
    for h in horizons:
        sub = df_basis[df_basis['hour']==hour].dropna(
            subset=['s_momentum', f'fwd_{h}s'])
        ic  = sub['s_momentum'].corr(sub[f'fwd_{h}s'])
        print(f'{ic:>8.3f}', end='')
    print()

=== Full Horizon IC Analysis — HDFCBANK ===

Signal                    60s   120s   300s   600s   900s  1800s  3600s
----------------------------------------------------------------------
Weighted Mid Basis      0.027  0.027  0.018  0.031  0.026  0.028  0.022
Price Momentum         -0.049 -0.049 -0.041 -0.032 -0.044 -0.075 -0.042
OI Momentum            -0.020 -0.019 -0.025 -0.026 -0.027 -0.017 -0.017
Price-OI Divergence     0.005  0.004  0.012  0.015  0.013 -0.005  0.005

=== basis_wmid_z IC by Hour and Horizon ===
Hour       60s    120s    300s    600s    900s   1800s   3600s
--------------------------------------------------------------
9:00    -0.002   0.011   0.001   0.022  -0.001  -0.066  -0.033
10:00     0.021   0.033   0.019   0.039   0.033   0.042   0.093
11:00     0.039   0.029   0.019   0.019   0.033   0.058   0.020
12:00    -0.007   0.004   0.005  -0.042  -0.051  -0.087  -0.086
13:00     0.015  -0.016  -0.055  -0.003  -0.023  -0.062  -0.102
14:00     0.077   0.076   0.099   

In [7]:
# ── Boost basis_wmid signal ────────────────────────────────────────

# First align HDFCBANK + ICICIBANK + BANKNIFTY on same timestamps
print('Aligning timestamps...')

# Build wmid signal for each instrument
def get_wmid_signal(df, name):
    sig_dfs = []
    for d in sorted(df['date'].unique()):
        sub = df[df['date']==d].copy().reset_index(drop=True)
        sub['mid']       = (sub['bid_p1'] + sub['ask_p1']) / 2
        sub['wmid_raw']  = sub['weighted_mid'] - sub['mid']
        sub['wmid_z']    = (sub['wmid_raw'] /
                            (sub['wmid_raw'].rolling(300).std()+1e-9))
        sub['wmid_ma']   = sub['wmid_z'].rolling(60).mean()  # persistence
        sub['vol_ratio'] = sub['tick_count'] / (
                           sub['tick_count'].rolling(300).mean()+1e-9)
        sub['hour']      = sub['ist_sec'] // 3600
        sub['fwd_1800s'] = sub['close'].shift(-1800) - sub['close']
        sub['fwd_300s']  = sub['close'].shift(-300)  - sub['close']
        sig_dfs.append(sub[['date','ist_sec','hour','ts_sec',
                             'wmid_z','wmid_ma','vol_ratio',
                             'close','fwd_1800s','fwd_300s'
                             ]].assign(symbol=name))
    return pd.concat(sig_dfs, ignore_index=True)

df_h = get_wmid_signal(data['HDFCBANK'],  'HDFCBANK')
df_i = get_wmid_signal(data['ICICIBANK'], 'ICICIBANK')
df_b = get_wmid_signal(data['BANKNIFTY'], 'BANKNIFTY')

# Merge on date + ts_sec
df_h2 = df_h.rename(columns={'wmid_z':'h_wmid','wmid_ma':'h_wmid_ma',
                               'vol_ratio':'h_vol','close':'h_close',
                               'fwd_1800s':'fwd_1800s','fwd_300s':'fwd_300s'})
df_i2 = df_i[['date','ts_sec','wmid_z','wmid_ma','vol_ratio']].rename(
    columns={'wmid_z':'i_wmid','wmid_ma':'i_wmid_ma','vol_ratio':'i_vol'})
df_b2 = df_b[['date','ts_sec','wmid_z','wmid_ma','vol_ratio']].rename(
    columns={'wmid_z':'b_wmid','wmid_ma':'b_wmid_ma','vol_ratio':'b_vol'})

df_merged = df_h2.merge(df_i2, on=['date','ts_sec'], how='inner')
df_merged = df_merged.merge(df_b2, on=['date','ts_sec'], how='inner')
print(f'Merged bars: {len(df_merged):,}')

# ── Test boosted signals ───────────────────────────────────────────
# Signal 1: HDFCBANK wmid only (baseline)
df_merged['sig1'] = df_merged['h_wmid']

# Signal 2: Persistence (sustained wmid)
df_merged['sig2'] = df_merged['h_wmid_ma']

# Signal 3: Volume confirmation
df_merged['sig3'] = df_merged['h_wmid'] * np.where(
    df_merged['h_vol'] > 1.5, 1.5, df_merged['h_vol'])

# Signal 4: Sector confirmation (HDFCBANK + ICICIBANK)
df_merged['sig4'] = (df_merged['h_wmid'] + df_merged['i_wmid']) / 2

# Signal 5: Full sector (HDFCBANK + ICICIBANK + BANKNIFTY)
df_merged['sig5'] = (df_merged['h_wmid'] +
                     df_merged['i_wmid'] +
                     df_merged['b_wmid']) / 3

# Signal 6: Persistence + sector
df_merged['sig6'] = (df_merged['h_wmid_ma'] + df_merged['i_wmid_ma']) / 2

# Signal 7: Volume + sector + persistence
df_merged['sig7'] = (
    0.4*df_merged['h_wmid_ma'] +
    0.3*df_merged['i_wmid_ma'] +
    0.3*df_merged['b_wmid_ma']
) * np.where(df_merged['h_vol'] > 1.2, 1.2, df_merged['h_vol'])

val = df_merged.dropna(subset=['fwd_1800s','fwd_300s'])

print()
print('=== Signal Comparison ===')
print(f'{"Signal":<30} {"IC_300s":>8} {"IC_1800s":>8}')
print('-'*48)
descs = {
    'sig1': 'HDFC wmid only (baseline)',
    'sig2': 'HDFC wmid persistence (60s MA)',
    'sig3': 'HDFC wmid × volume',
    'sig4': 'HDFC + ICICI sector',
    'sig5': 'HDFC + ICICI + BANKNIFTY',
    'sig6': 'Sector persistence',
    'sig7': 'Sector + persistence + vol',
}
for sig, desc in descs.items():
    ic300  = val[sig].corr(val['fwd_300s'])
    ic1800 = val[sig].corr(val['fwd_1800s'])
    print(f'{desc:<30} {ic300:>8.4f} {ic1800:>8.4f}')

# Best signal by hour
print()
print('=== Best Signal by Hour ===')
print(f'{"Hour":<6}', end='')
for sig in ['sig1','sig4','sig6','sig7']:
    print(f'{sig:>10}', end='')
print()
print('-'*46)
for h in range(9, 15):
    mask = val['hour'] == h
    print(f'{h}:00  ', end='')
    for sig in ['sig1','sig4','sig6','sig7']:
        ic = val[mask][sig].corr(val[mask]['fwd_1800s'])
        print(f'{ic:>10.4f}', end='')
    print()

Aligning timestamps...
Merged bars: 92,374

=== Signal Comparison ===
Signal                          IC_300s IC_1800s
------------------------------------------------
HDFC wmid only (baseline)        0.0205   0.0362
HDFC wmid persistence (60s MA)   0.0260   0.0709
HDFC wmid × volume               0.0192   0.0341
HDFC + ICICI sector             -0.0030  -0.0157
HDFC + ICICI + BANKNIFTY        -0.0193  -0.0385
Sector persistence              -0.0184  -0.0353
Sector + persistence + vol      -0.0437  -0.0590

=== Best Signal by Hour ===
Hour        sig1      sig4      sig6      sig7
----------------------------------------------
9:00     -0.0508   -0.1411   -0.3045   -0.3817
10:00      0.0634   -0.0310   -0.0497   -0.0841
11:00      0.0549   -0.1124   -0.2240   -0.1434
12:00     -0.0809   -0.0530   -0.0914   -0.1836
13:00     -0.0702   -0.1051   -0.2015   -0.0826
14:00      0.1212    0.0916    0.1715    0.0119


In [8]:
# Build volume bars from 1-second data
def build_volume_bars(df, volume_threshold=5000):
    """
    Build bars every time cumulative volume crosses threshold
    Much better statistical properties than time bars
    """
    bars   = []
    cum_vol = 0
    bar_open = None
    bar_high = -np.inf
    bar_low  = np.inf
    bar_start= None
    ticks    = 0
    
    for _, row in df.iterrows():
        if bar_open is None:
            bar_open  = row['close']
            bar_start = row['ist_sec']
        
        bar_high = max(bar_high, row['high'])
        bar_low  = min(bar_low,  row['low'])
        cum_vol += abs(row['volume_delta']) + 1
        ticks   += 1
        
        if cum_vol >= volume_threshold:
            bars.append({
                'ist_sec':    bar_start,
                'open':       bar_open,
                'high':       bar_high,
                'low':        bar_low,
                'close':      row['close'],
                'volume':     cum_vol,
                'tick_count': ticks,
                'duration':   row['ist_sec'] - bar_start,
                'imbalance':  row['imbalance_last'],
                'spread':     row['spread_mean'],
                'date':       row['date'],
            })
            # Reset
            cum_vol  = 0
            bar_open = None
            bar_high = -np.inf
            bar_low  = np.inf
            bar_start= None
            ticks    = 0
    
    return pd.DataFrame(bars)

# Test on HDFCBANK
print('Building volume bars for HDFCBANK...')
vol_bar_dfs = []
for d in sorted(data['HDFCBANK']['date'].unique()):
    day = data['HDFCBANK'][data['HDFCBANK']['date']==d].copy()
    day['ist_sec'] = (day['ts_sec'] + 19800) % 86400
    day = day[(day['ist_sec'] >= 9*3600+15*60) &
              (day['ist_sec'] <= 15*3600+30*60)].reset_index(drop=True)
    day['date'] = d
    vbars = build_volume_bars(day, volume_threshold=5000)
    vol_bar_dfs.append(vbars)
    print(f'  {d}: {len(day)} time bars → {len(vbars)} volume bars')

df_vbars = pd.concat(vol_bar_dfs, ignore_index=True)
print(f'\nTotal volume bars: {len(df_vbars):,}')
print(f'Avg duration: {df_vbars["duration"].mean():.0f}s per bar')
print(f'Avg ticks: {df_vbars["tick_count"].mean():.0f} ticks per bar')

Building volume bars for HDFCBANK...
  2026-09-22: 14885 time bars → 786 volume bars
  2026-09-23: 15361 time bars → 1004 volume bars
  2026-09-24: 15660 time bars → 1795 volume bars
  2026-09-25: 15786 time bars → 1677 volume bars
  2026-09-28: 19682 time bars → 2019 volume bars
  2026-09-30: 21140 time bars → 878 volume bars
  2026-10-01: 20712 time bars → 932 volume bars

Total volume bars: 9,091
Avg duration: 16s per bar
Avg ticks: 14 ticks per bar


In [9]:
# ── VPIN Implementation ────────────────────────────────────────────
# Easley, Lopez de Prado, O'Hara (2012)
# VPIN = |V_buy - V_sell| / V_total
# Estimated using tick rule: price up → buy volume, price down → sell volume

def compute_vpin(df, window=50):
    """
    Compute VPIN on volume bars
    window: number of bars for rolling calculation
    """
    df = df.copy()
    
    # Classify volume as buy or sell using tick rule
    df['price_chg'] = df['close'].diff()
    df['buy_vol']   = np.where(df['price_chg'] >= 0,
                               df['volume'], 0)
    df['sell_vol']  = np.where(df['price_chg'] < 0,
                               df['volume'], 0)
    
    # Rolling VPIN
    df['vpin'] = (
        (df['buy_vol'].rolling(window).sum() -
         df['sell_vol'].rolling(window).sum()).abs() /
        (df['volume'].rolling(window).sum() + 1e-9)
    )
    
    # VPIN z-score (relative to recent history)
    df['vpin_z'] = (
        (df['vpin'] - df['vpin'].rolling(window*2).mean()) /
        (df['vpin'].rolling(window*2).std() + 1e-9)
    )
    
    return df

# ── Triple Barrier Labeling ────────────────────────────────────────
def triple_barrier_label(prices, idx, tp_pct=0.0015, sl_pct=0.0010,
                          max_bars=20):
    """
    Label each bar using triple barrier method
    tp_pct:   take profit barrier (% of price)
    sl_pct:   stop loss barrier (% of price)
    max_bars: time barrier (max bars to hold)
    
    Returns: +1 (TP hit), -1 (SL hit), 0 (time barrier)
    """
    labels = np.zeros(len(prices))
    
    for i in range(len(prices) - max_bars):
        entry = prices[i]
        tp    = entry * (1 + tp_pct)
        sl    = entry * (1 - sl_pct)
        
        label = 0  # default: time barrier
        for j in range(1, max_bars + 1):
            if i + j >= len(prices):
                break
            p = prices[i + j]
            if p >= tp:
                label = 1   # TP hit
                break
            elif p <= sl:
                label = -1  # SL hit
                break
        labels[i] = label
    
    return labels

# ── Apply to HDFCBANK volume bars ─────────────────────────────────
print('Computing VPIN and Triple Barrier labels...')

processed_dfs = []
for d in sorted(df_vbars['date'].unique()):
    day = df_vbars[df_vbars['date']==d].copy().reset_index(drop=True)
    
    if len(day) < 60:
        continue
    
    # VPIN
    day = compute_vpin(day, window=50)
    
    # Triple barrier labels
    prices = day['close'].values
    day['label_tight']  = triple_barrier_label(prices, None,
                           tp_pct=0.0010, sl_pct=0.0007, max_bars=20)
    day['label_normal'] = triple_barrier_label(prices, None,
                           tp_pct=0.0015, sl_pct=0.0010, max_bars=30)
    day['label_wide']   = triple_barrier_label(prices, None,
                           tp_pct=0.0025, sl_pct=0.0015, max_bars=50)
    
    # Signal on volume bars
    day['ret']      = day['close'].pct_change()
    day['wmid_z']   = (day['spread'] /
                       (day['spread'].rolling(20).std() + 1e-9))
    day['mom']      = day['close'].pct_change(5)
    day['vol_z']    = (day['volume'] /
                       (day['volume'].rolling(20).mean() + 1e-9))
    
    # Forward returns (in bars not seconds)
    day['fwd_10bar'] = day['close'].shift(-10) - day['close']
    day['fwd_20bar'] = day['close'].shift(-20) - day['close']
    day['fwd_30bar'] = day['close'].shift(-30) - day['close']
    
    day['hour'] = day['ist_sec'] // 3600
    processed_dfs.append(day)

df_vol = pd.concat(processed_dfs, ignore_index=True)
print(f'Total bars: {len(df_vol):,}')

# ── Check label distribution ───────────────────────────────────────
print()
print('=== Triple Barrier Label Distribution ===')
for col in ['label_tight','label_normal','label_wide']:
    val = df_vol[df_vol[col] != 0][col]
    pos = (val == 1).mean() * 100
    neg = (val == -1).mean() * 100
    print(f'{col:<16}: TP={pos:.1f}%  SL={neg:.1f}%  '
          f'inconclusive={(df_vol[col]==0).mean()*100:.1f}%')

# ── VPIN analysis ─────────────────────────────────────────────────
print()
print('=== VPIN Statistics ===')
print(f'VPIN mean:   {df_vol["vpin"].mean():.4f}')
print(f'VPIN std:    {df_vol["vpin"].std():.4f}')
print(f'VPIN > 0.5:  {(df_vol["vpin"] > 0.5).mean()*100:.1f}%')
print(f'VPIN > 0.7:  {(df_vol["vpin"] > 0.7).mean()*100:.1f}%')

# ── IC on volume bars ─────────────────────────────────────────────
print()
print('=== IC on Volume Bars ===')
val = df_vol.dropna(subset=['wmid_z','fwd_20bar'])

signals = {
    'wmid_z':              'WMid basis',
    'mom':                 'Momentum',
    'vpin_z':              'VPIN',
    'vol_z':               'Volume ratio',
}

# Add combined
df_vol['combined'] = (
    0.4*df_vol['wmid_z'].fillna(0) +
    0.3*df_vol['vpin_z'].fillna(0) +
    0.3*df_vol['mom'].fillna(0)
)
signals['combined'] = 'WMid + VPIN + Mom'

print(f'{"Signal":<22} {"IC_10bar":>10} {"IC_20bar":>10} {"IC_30bar":>10}')
print('-'*54)
for sig, name in signals.items():
    v      = df_vol.dropna(subset=[sig,'fwd_20bar'])
    ic10   = v[sig].corr(v['fwd_10bar'])
    ic20   = v[sig].corr(v['fwd_20bar'])
    ic30   = v[sig].corr(v['fwd_30bar'])
    print(f'{name:<22} {ic10:>10.4f} {ic20:>10.4f} {ic30:>10.4f}')

# By hour
print()
print('=== Combined Signal IC by Hour (volume bars) ===')
print(f'{"Hour":<6} {"IC_10bar":>10} {"IC_20bar":>10} {"n_bars":>8}')
print('-'*36)
v2 = df_vol.dropna(subset=['combined','fwd_20bar'])
for h in range(9, 15):
    mask  = v2['hour'] == h
    ic10  = v2[mask]['combined'].corr(v2[mask]['fwd_10bar'])
    ic20  = v2[mask]['combined'].corr(v2[mask]['fwd_20bar'])
    print(f'{h}:00   {ic10:>10.4f} {ic20:>10.4f} {mask.sum():>8}')

Computing VPIN and Triple Barrier labels...
Total bars: 9,091

=== Triple Barrier Label Distribution ===
label_tight     : TP=38.2%  SL=61.8%  inconclusive=41.3%
label_normal    : TP=34.4%  SL=65.6%  inconclusive=49.4%
label_wide      : TP=27.8%  SL=72.2%  inconclusive=61.1%

=== VPIN Statistics ===
VPIN mean:   0.2506
VPIN std:    0.1716
VPIN > 0.5:  9.6%
VPIN > 0.7:  1.2%

=== IC on Volume Bars ===
Signal                   IC_10bar   IC_20bar   IC_30bar
------------------------------------------------------
WMid basis                -0.0035    -0.0312    -0.0535
Momentum                  -0.0750    -0.0644    -0.0672
VPIN                      -0.0440    -0.0249    -0.0003
Volume ratio              -0.0067    -0.0008     0.0012
WMid + VPIN + Mom         -0.0456    -0.0451    -0.0414

=== Combined Signal IC by Hour (volume bars) ===
Hour     IC_10bar   IC_20bar   n_bars
------------------------------------
9:00      -0.1204    -0.0962     1065
10:00      -0.0784    -0.0816     1624
11: